# Coleta de Dados

Coleta e preparo da base de dados.

- **Etapa 1:** preços e volatilidade realizada (Yahoo Finance)
- **Etapa 2:** SVI semanal por ticker (Google Trends)
- **Etapa 3:** teste de estacionariedade ADF
- **Etapa 4:** construção da base final (merge preços + SVI)

Os arquivos gerados são salvos em `./data/` e carregados pelo notebook `02_modelos.ipynb`.

In [1]:
import time
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import yfinance as yf

from pytrends.request import TrendReq
from datetime import datetime, timedelta
from statsmodels.tsa.stattools import adfuller


In [ ]:
# ──────────────────────────────────────────────────────────────
# CONFIGURAÇÕES
# ──────────────────────────────────────────────────────────────
ANOS_HISTORICO    = 5
VOLUME_MIN_DIARIO = 1
SEMANAS           = 260
DATA_FIM          = datetime(2026, 3, 15).strftime("%Y-%m-%d")
DATA_INICIO       = (datetime(2026, 3, 15) - timedelta(weeks=SEMANAS)).strftime("%Y-%m-%d")
PAUSA_TRENDS      = 12     # segundos entre requisições
CAMINHO  = "./data/"

In [ ]:
(datetime.strptime(DATA_FIM, "%Y-%m-%d") + timedelta(days=5)).strftime("%Y-%m-%d")


'2026-03-20'

## Dados financeiros

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 1 — PREÇOS
# ──────────────────────────────────────────────────────────────

# Todos os ticker
tickers = pd.read_csv("./data/acoes-listadas-b3.csv")
TERMOS = list(tickers.Ticker)
tickers_yf = [t + ".SA" for t in TERMOS]

dados_raw = yf.download(
    tickers_yf, start=DATA_INICIO, end=(datetime.strptime(DATA_FIM, "%Y-%m-%d") + timedelta(days=5)).strftime("%Y-%m-%d"), # Necessário para fazer o merge com google trends
    auto_adjust=True, progress=True,
)

registros_acoes = []
for ticker_yf in tickers_yf:

    try:
        close  = dados_raw["Close"][ticker_yf].dropna()
        volume = dados_raw["Volume"][ticker_yf].dropna()

        if len(close) < ANOS_HISTORICO * 252 * 0.85:
            print(f"  [SKIP] {ticker_yf} — histórico insuficiente ({len(close)} dias)")
            continue

        vol_fin = (volume * close.mean()).mean()
        if vol_fin < VOLUME_MIN_DIARIO:
            print(f"  [SKIP] {ticker_yf} — liquidez insuficiente")
            continue

        registros_acoes.append({
            "ticker_yf": ticker_yf,
            "dias": len(close),
            "vol_fin_medio": round(vol_fin, 2),
        })
        print(f"  [OK] {ticker_yf} | Vol R${vol_fin:>14,.0f}")

    except Exception as e:
        print(f"  [ERRO] {ticker_yf}: {e}")

df_acoes = pd.DataFrame(registros_acoes)
df_acoes.to_csv("./data/acoes_elegiveis.csv", index=False)
print(f"\n✅ {len(df_acoes)} ações elegíveis → acoes_elegiveis.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 2 — VOLATILIDADE REALIZADA
# ──────────────────────────────────────────────────────────────

series_semanais = []
for _, row in df_acoes.iterrows():
    try:
        close  = dados_raw["Close"][row["ticker_yf"]].dropna()
        volume = dados_raw["Volume"][row["ticker_yf"]].dropna()

        df_d = pd.DataFrame({"close": close, "volume": volume})
        df_d.index = pd.to_datetime(df_d.index)

        df_d["log_ret"] = np.log(df_d["close"] / df_d["close"].shift(1))

        # Agrega semanalmente (semana fecha na sexta)
        wk = df_d.resample("W-FRI").agg(
            preco_abertura   = ("close",   "first"),
            preco_fechamento = ("close",   "last"),
            volume_total     = ("volume",  "sum"),
            n_preços         = ("log_ret", "count"), # nº de pregões na semana
            rv               = ("log_ret", lambda x: (x**2).sum()),  # RV = Σ r²
            ret_semanal      = ("log_ret", "sum"),
        ).dropna(subset=["rv"])

        wk["ticker_yf"] = row["ticker_yf"]
        wk["ticker_b3"] = row["ticker_yf"].replace(".SA", "")
        series_semanais.append(wk.reset_index().rename(columns={"Date": "semana"}))

    except Exception as e:
        print(f"  [ERRO preços {row['ticker_yf']}]: {e}")

df_precos = pd.concat(series_semanais, ignore_index=True)

# Retirar ações que não possuem o número de semanas correto
ticker_counts = df_precos.groupby("ticker_b3")["semana"].count()
tickers_corretos = ticker_counts[ticker_counts == SEMANAS].index
df_precos = df_precos[df_precos["ticker_b3"].isin(tickers_corretos)]


df_precos.to_csv("./data/precos_semanais.csv", index=False)
print(f"✅ {len(df_precos)} linhas → precos_semanais.csv")

## GOOGLE TRENDS: SVI POR TICKER

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 3 — GOOGLE TRENDS: SVI POR TICKER
# ──────────────────────────────────────────────────────────────

pytrends = TrendReq(hl="pt-BR", tz=-180)

def coletar_svi(ticker):
    """Retorna DataFrame semanal do SVI para um ticker ou None."""
    try:
        pytrends.build_payload(
            kw_list=[ticker],
            timeframe=f"{DATA_INICIO} {DATA_FIM}",
            geo="BR",
            gprop="",
        )
        df = pytrends.interest_over_time()
        if df.empty:
            return None
        df = df.drop(columns=["isPartial"], errors="ignore")
        df.columns = ["svi"]
        df.index.name = "semana"
        return df.reset_index()
    except Exception as e:
        print(f"    [ERRO '{ticker}']: {e}")
        return None


todos_trends = []
log_coleta   = []
tickers_validos = list(df_precos["ticker_b3"].unique())

for ticker in tickers_validos:

    print(f"\n  {ticker}")
    df_t = coletar_svi(ticker)
    time.sleep(PAUSA_TRENDS)

    if df_t is None:
        print("VAZIO")
        log_coleta.append({
            "ticker": ticker, "status": "vazio",
            "semanas": 0, "svi_media": None, "svi_std": None,
        })
        continue

    svi_media = round(df_t["svi"].mean(), 2)
    svi_std   = round(df_t["svi"].std(), 2)
    svi_zeros = (df_t["svi"] == 0).mean()
    print(f"OK | {len(df_t)} sem | média={svi_media:.1f} "
          f"std={svi_std:.1f} zeros={svi_zeros:.0%}")

    for _, tr in df_t.iterrows():
        todos_trends.append({
            "semana": tr["semana"],
            "ticker_b3": ticker,
            "svi":    tr["svi"],
        })

    log_coleta.append({
        "ticker":       ticker,
        "status":       "ok",
        "semanas":      len(df_t),
        "svi_media":    svi_media,
        "svi_std":      svi_std,
        "svi_zeros_pct": round(svi_zeros * 100, 1),
    })


df_trends = pd.DataFrame(todos_trends)
df_log    = pd.DataFrame(log_coleta)

# Calcular % de zeros por ticker no df_trends
zeros_pct = df_trends.groupby("ticker_b3")["svi"].apply(lambda x: (x == 0).mean())
tickers_sem_sinal = zeros_pct[zeros_pct == 1.0].index
df_trends = df_trends[~df_trends["ticker_b3"].isin(tickers_sem_sinal)]

df_trends.to_csv("./data/google_trends_svi.csv", index=False)
df_log.to_csv("./data/log_coleta_svi.csv", index=False)

print(f"\n✅ {len(df_trends)} linhas → google_trends_svi.csv")
print(f"✅ Log de coleta    → log_coleta_svi.csv")

## ESTACIONARIEDADE DO SVI (ADF POR TICKER)

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 4 — ESTACIONARIEDADE DO SVI (ADF POR TICKER)
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

NIVEL_SIGNIFICANCIA = 0.05

def teste_adf(serie, nome=""):
    """
    Roda o ADF e retorna dict com estatística, p-valor e conclusão.
    Usa seleção automática de lags (método 'AIC').
    """
    serie = serie.dropna()
    if len(serie) < 10:
        return {"stat": None, "pvalor": None, "estacionaria": None, "obs": "série curta"}

    stat, pvalor, _, _, valores_criticos, _ = adfuller(serie, autolag="AIC")
    estacionaria = pvalor < NIVEL_SIGNIFICANCIA

    # print(f"      {'[OK]' if estacionaria else '[RAIZ UNITÁRIA]'} {nome}: "
          # f"stat={stat:.3f}  p={pvalor:.4f}  {'estacionária' if estacionaria else 'não estacionária'}")

    return {
        "stat":         round(stat, 4),
        "pvalor":       round(pvalor, 4),
        "cv_1pct":      round(valores_criticos["1%"], 3),
        "cv_5pct":      round(valores_criticos["5%"], 3),
        "cv_10pct":     round(valores_criticos["10%"], 3),
        "estacionaria": estacionaria,
        "obs":          "",
    }


resultados_adf = []

for ticker, grupo in df_trends.groupby("ticker_b3"):

    # print(f"\n  {ticker}")
    svi = grupo.sort_values("semana")["svi"].reset_index(drop=True)

    # ── ADF no nível ──────────────────────────────────────────
    adf_nivel = teste_adf(svi, "SVI nível")

    # ── Primeira diferença e ADF ──────────────────────────────
    svi_diff  = svi.diff()
    adf_diff  = teste_adf(svi_diff, "SVI Δ(1)")

    resultados_adf.append({
        "ticker_b3":                   ticker,
        # nível
        "svi_adf_stat":             adf_nivel["stat"],
        "svi_adf_pvalor":           adf_nivel["pvalor"],
        "svi_estacionario":         adf_nivel["estacionaria"],
        # primeira diferença
        "svi_diff_adf_stat":        adf_diff["stat"],
        "svi_diff_adf_pvalor":      adf_diff["pvalor"],
        "svi_diff_estacionario":    adf_diff["estacionaria"],
    })

    # ── Adiciona coluna svi_diff ao df_trends ─────────────────
    df_trends.loc[df_trends["ticker_b3"] == ticker, "svi_diff"] = svi_diff.values


# ── Bases finais ──────────────────────────────────────────────
df_adf = pd.DataFrame(resultados_adf)

df_adf.to_csv("./data/log_adf_svi.csv", index=False)
df_trends.to_csv("./data/google_trends_svi.csv", index=False)

# ── Resumo ────────────────────────────────────────────────────
n_nivel_ok = df_adf["svi_estacionario"].sum()
n_diff_ok  = df_adf["svi_diff_estacionario"].sum()
n          = len(df_adf)

print(f"\n{'─'*55}")
print(f"  SVI nível estacionário:      {n_nivel_ok}/{n} tickers")
print(f"  SVI Δ(1)  estacionário:      {n_diff_ok}/{n} tickers")
print(f"\n✅ google_trends_svi.csv  → svi + svi_diff por semana")
print(f"✅ log_adf_svi.csv        → resultado ADF por ticker")

## Base de dados final

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 5 — MERGE: PREÇOS + TRENDS
# Uma linha por (semana, ticker)
# ──────────────────────────────────────────────────────────────

df_trends["semana"] = pd.to_datetime(df_trends["semana"])
df_precos["semana"] = pd.to_datetime(df_precos["semana"])

# Alinha domingo (Trends) → sexta (preços)
df_trends_temp = df_trends.copy()
df_trends_temp["semana"] = df_trends_temp["semana"] + pd.offsets.Week(weekday=4)

df_final = pd.merge(
    df_precos,
    df_trends_temp[["semana","ticker_b3","svi","svi_diff"]],
    on=["semana","ticker_b3"],
    how="inner",
).sort_values(["ticker_b3","semana"]).reset_index(drop=True)

# Selecionar apenas o ticker com SVI Diff estacionário
df_final = df_final[df_final["ticker_b3"].isin(df_adf[df_adf['svi_diff_estacionario'] == True]["ticker_b3"])]

# Retirar os valores NaN da primeira diferença do svi
df_final = df_final[df_final["svi_diff"].notna()]

# Retirar ativos com número de semanas errada
ticker_counts = df_final.groupby("ticker_b3")["semana"].count()
tickers_ok = ticker_counts[ticker_counts == (SEMANAS - 1)].index
df_final = df_final[df_final["ticker_b3"].isin(tickers_ok)]

df_final.to_csv("./data/base_final_tcc.csv", index=False)

print(f"✅ BASE FINAL → base_final_tcc.csv")

In [6]:
df_trends = pd.read_csv("./data/google_trends_svi.csv")
df_precos = pd.read_csv("./data/precos_semanais.csv")
df_trends["semana"] = pd.to_datetime(df_trends["semana"])
df_precos["semana"] = pd.to_datetime(df_precos["semana"])

In [12]:
df_trends["semana_offset"] = df_trends["semana"] + pd.offsets.Week(weekday=4)
df_trends[df_trends["ticker_b3"] == 'PETR4']

,semana,ticker_b3,svi,svi_diff,semana_offset
0,2021-03-21,PETR4,35,NaN,2021-03-26
1,2021-03-28,PETR4,31,-4.0,2021-04-02
2,2021-04-04,PETR4,31,0.0,2021-04-09
3,2021-04-11,PETR4,31,0.0,2021-04-16
4,2021-04-18,PETR4,30,-1.0,2021-04-23
...,...,...,...,...,...
256,2026-02-15,PETR4,16,-8.0,2026-02-20
257,2026-02-22,PETR4,23,7.0,2026-02-27
258,2026-03-01,PETR4,51,28.0,2026-03-06
259,2026-03-08,PETR4,46,-5.0,2026-03-13


In [11]:
df_precos[df_precos["ticker_b3"] == 'PETR4']

,semana,preco_abertura,preco_fechamento,volume_total,n_preços,rv,ret_semanal,ticker_yf,ticker_b3
0,2021-03-26,6.754333,6.737103,304725400.0,4,0.001364,-0.002554,PETR4.SA,PETR4
1,2021-04-02,6.843357,6.860588,207901100.0,4,0.000448,0.018163,PETR4.SA,PETR4
2,2021-04-09,6.903664,6.791666,253006600.0,5,0.000203,-0.010097,PETR4.SA,PETR4
3,2021-04-16,6.860588,6.812458,318802600.0,5,0.000799,0.003057,PETR4.SA,PETR4
4,2021-04-23,7.207254,7.032118,398425600.0,4,0.003562,0.031735,PETR4.SA,PETR4
...,...,...,...,...,...,...,...,...,...
255,2026-02-13,37.320000,36.889999,174901900.0,5,0.001405,0.006527,PETR4.SA,PETR4
256,2026-02-20,37.189999,37.970001,132581500.0,3,0.000357,0.028856,PETR4.SA,PETR4
257,2026-02-27,38.590000,39.330002,193055500.0,5,0.000943,0.035191,PETR4.SA,PETR4
258,2026-03-06,41.130001,42.160000,368962000.0,5,0.003425,0.069484,PETR4.SA,PETR4
